# HW8: user-history Encoder + identical user-kNN
**RQ:** Улучшает ли frozen Qwen-кодирование профиля прошлых покупок пользователя nDCG@10 по сравнению со структурированным кодированием тех же входных данных при неизменном user-kNN?

Notebook подготовлен; сохранённых результатов Qwen нет. Запуск GPU/inference — только после отдельного подтверждения автора. Для GPU выберите Runtime → Change runtime type → T4 GPU. CPU поддерживается явно, но может быть медленным.

Перед Colab локально: `python hw8/src/bundle.py`. Загрузите созданный `hw8/outputs/workflow_bundle.zip` в следующей ячейке. Он содержит собственный код и неизменённый локальный starter с проверкой SHA-256; данных, моделей и результатов в нём нет. Это позволяет работать и с ещё не опубликованной веткой. Не загружайте весь рабочий каталог.

FAR-Trans: Sanz-Cruzado, Droukas, McCreadie (2024), DOI 10.5525/gla.researchdata.1658, CC BY 4.0. Данные загружаются отдельно пользователем.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess, shutil
from google.colab import files
WORK = Path('/content/far_trans_hw8')
WORK.mkdir(parents=True, exist_ok=True)
def safe_extract(archive, destination):
    destination = destination.resolve()
    for member in archive.infolist():
        path = (destination / member.filename).resolve()
        if not path.is_relative_to(destination):
            raise ValueError('Unsafe archive path')
    archive.extractall(destination)
uploaded = files.upload()  # Select workflow_bundle.zip only
assert list(uploaded) == ['workflow_bundle.zip']
safe_extract(zipfile.ZipFile('workflow_bundle.zip'), WORK)
sys.path.insert(0, str(WORK / 'hw8/src'))
from protocol import load_starter
load_starter(WORK / 'hw8/reference/starter/starter.py')
print('Workflow uploaded; starter hash verified.')


## Совместимые зависимости
Используйте Python 3.11/3.12 runtime. После установки **перезапустите session/kernel** через меню Runtime, затем продолжайте с ячейки восстановления путей. Файлы в `/content` сохраняются при restart kernel (но не при удалении runtime). Конфликты предустановленных неиспользуемых Colab-пакетов не меняют pins эксперимента; проверка ниже проверяет реально загруженные версии.

In [ ]:
assert sys.version_info[:2] in [(3, 11), (3, 12)], sys.version
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'hw8/requirements-colab.txt')], check=True)
print('Now restart the kernel/session, then run the next cell.')


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, importlib.metadata
from google.colab import files
WORK = Path('/content/far_trans_hw8')
sys.path.insert(0, str(WORK / 'hw8/src'))
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['HF_HOME'] = '/content/hw8_model_cache'
import torch
from protocol import load_starter
load_starter(WORK / 'hw8/reference/starter/starter.py')
for line in (WORK / 'hw8/requirements-colab.txt').read_text().splitlines():
    if '==' in line and not line.startswith('#'):
        package, pinned = line.split('==')
        actual = importlib.metadata.version(package)
        assert actual.split('+')[0] == pinned, (package, actual, pinned)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Python:', sys.version, 'device:', DEVICE)
print(torch.cuda.get_device_name() if DEVICE == 'cuda' else 'CPU; explicitly select CPU below if intended')
OUT = WORK / 'hw8/outputs/colab'
def run(action, *extra):
    subprocess.run([sys.executable, str(WORK / 'hw8/src/experiment.py'), action,
                    '--data-dir', str(DATA), '--output-dir', str(OUT), *extra], check=True)


## FAR-Trans: ZIP или четыре CSV
Загрузите официальный FAR-Trans.zip либо одновременно `transactions.csv`, `asset_information.csv`, `customer_information.csv`, `close_prices.csv`. Значения данных не печатаются в сохранённые ячейки. SHA-256 фиксируется starter; при отличии локального протокола сравнение остановится. Для сопоставления с локальным запуском можно также загрузить его `protocol.json` в отдельной ячейке ниже.

In [ ]:
from protocol import SOURCE_FILES
RAW = WORK / 'data/raw/upload'
RAW.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
for name, payload in uploaded.items():
    if name.lower().endswith('.zip'):
        import io
        with zipfile.ZipFile(io.BytesIO(payload)) as archive:
            for member in archive.infolist():
                base = Path(member.filename).name
                if base in SOURCE_FILES:
                    target = RAW / base
                    assert not target.exists(), 'Use a clean data upload folder'
                    target.write_bytes(archive.read(member))
    else:
        assert name in SOURCE_FILES
        (RAW / name).write_bytes(payload)
del uploaded
assert all((RAW / name).is_file() for name in SOURCE_FILES)
DATA = RAW
print('Four source CSVs ready.')


In [ ]:
run('baseline')  # Unchanged starter, default 100-user protocol, all FOUR methods
run('local-check')  # Repeated exact scorer, past-history counts, Top-10 and independent nDCG
print((OUT / 'comparison.md').read_text())


In [ ]:
# Upload local hw8/outputs/local/baseline/protocol.json to compare FULL protocol.
uploaded = files.upload()
assert len(uploaded) == 1
local_protocol = json.loads(next(iter(uploaded.values())))
colab_protocol = json.loads((OUT / 'baseline/protocol.json').read_text())
assert local_protocol == colab_protocol, 'Protocol mismatch: investigate before inference'
print('Local / Colab protocol exactly identical:', colab_protocol['protocol_hash'])


## Encoder run — только после отдельного разрешения
Следующая ячейка по умолчанию заблокирована. После подтверждения автора измените `AUTHOR_APPROVED_QWEN_RUN` на `True`. Загружается **Qwen/Qwen3-Embedding-0.6B**, revision **97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3**, без fine-tuning.

Профили всех as-of-known warm пользователей используются как общая база соседей; оцениваются ровно 100 пользователей starter. Профили: category/subcategory/sector/industry + count + recency (30/180 дней), без имён/ID. Полные строки лексикографического префикса укладываются в 512 токенов pinned tokenizer **до обеих веток**. One-hot получает те же сокращённые строки; B сохраняет всю допустимую прошлую историю.

Один scorer: exact cosine, k=20, без self, отрицательные cosine → 0, ties соседей по user ID, активов по ISIN, repeats allowed, popularity fallback при отсутствии положительных item scores. Параметры заданы заранее, не по June truth.

Inference: batch=8, eager attention, last non-padding token (left padding), L2 float32, CUDA float16 / CPU float32, eval/inference_mode. Точная английская инструкция и template сохраняются в manifest. Время загрузки модели, inference и scoring учитываются отдельно; четыре baseline имеют общий subprocess runtime. Векторы одинаковых текстов вычисляются один раз; число уникальных профилей и failures записываются. Ошибки не удаляют пользователей.

In [ ]:
AUTHOR_APPROVED_QWEN_RUN = False
ALLOW_CPU = False
assert AUTHOR_APPROVED_QWEN_RUN, 'Obtain separate author approval first.'
assert DEVICE == 'cuda' or ALLOW_CPU, 'Select GPU or explicitly allow CPU.'
run('encoders', '--allow-qwen-inference', '--device', DEVICE, '--batch-size', '8')


## Личная ручная проверка
Первая по протоколу непустая truth выбирается только для проверки **после** рекомендаций. Прочитайте Top-10 и relevant IDs, сверьте каждый binary hit с CSV будущих Buy в июне, затем самостоятельно сложите `hit/log2(rank+1)` и разделите на IDCG. Код ниже показывает все слагаемые и отдельный пересчёт. Автоматическая проверка агента не является вашей личной проверкой: после просмотра запишите собственное объяснение в отчёте. До Qwen run здесь доступны только выполненные baseline.

In [ ]:
check = json.loads((OUT / 'manual_check.json').read_text())
print('User:', check['user_id'], 'Relevant IDs:', check['relevant_ids'])
print('Source June Buy rows:', (OUT / 'manual_truth_source_rows.json').read_text())
for method, details in check['methods'].items():
    print(method, json.dumps(details, indent=2))
run('report')


## Сохранить и скачать outputs
Скачайте ZIP до отключения runtime. Распакуйте локально в `hw8/outputs/colab/`; данные и cache модели в архив не входят. Опционально сохраните outputs ZIP на свой Google Drive. Перед Git очистите все cell outputs через Edit → Clear all outputs; notebook не должен хранить dataset, векторы или результаты вызовов.

In [ ]:
archive = shutil.make_archive('/content/hw8_colab_outputs', 'zip', root_dir=OUT)
files.download(archive)
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/hw8')
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, destination / Path(archive).name)
